# h2ml Quickstart

This notebook demonstrates the h2ml pipeline on small built-in datasets:

1. **Classification** — breast cancer dataset (binary, 30 features)
2. **Regression** — diabetes dataset (10 features, y-transform sweep)
3. **Spatio-temporal** — synthetic geodata showing spatial CV, space-time local conformal intervals and the area of applicability
4. **Partial runs** — quick model screening
5. **Validation & robustness** — outer holdout, seed stability, nested CV and comparing selection metrics
6. **Persistence** — saving and reloading results and models

No external data required — everything runs with `sklearn.datasets` or a small synthetic generator.

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_diabetes

from h2ml import H2MLPipeline, PipelineConfig, PipelineData
from h2ml.plots import (
    cv_diagnostics,
    pipeline_scores,
    shap_dependence,
    shap_importance,
    shap_summary_plot,
)

---
## 1. Classification — breast cancer dataset

In [ ]:
data = load_breast_cancer()

store_clf = PipelineData(
    X=data.data.astype(np.float32),
    feature_names=list(data.feature_names),
    y=data.target.astype(np.float32),
)

print(store_clf)

In [ ]:
config_clf = PipelineConfig(
    task_type="classification",   # or "regression"; the TaskType enum is also accepted
    metric="AUC",
    n_splits=5,
    n_trials=30,
    verbose=True,
)

pipeline_clf = H2MLPipeline(config=config_clf)
result_clf = pipeline_clf.run(store_clf)

In [ ]:
# Results across all pipeline stages
result_clf.summary(metric="AUC_Test_Mean")

In [ ]:
# Features selected after step 2
print(f"Original features : {store_clf.n_features}")
print(f"Reduced features  : {result_clf.features_reduced.n_features}")
print(f"Kept              : {result_clf.features_reduced.feature_names}")

### Check the result

A winner can be statistically tied with other candidates, or rank lower on another metric. `cv_warnings` collects failed folds plus advisory `[metric]`, `[ranking]`, `[selection]`, `[transform]` and `[holdout]` flags; `ranking_robustness()` shows, per candidate, the paired-fold gap to the winner, whether it is `Indistinguishable` from it, and its rank under every metric.

In [ ]:
for w in result_clf.cv_warnings:
    print(w)

result_clf.ranking_robustness().head(8)

In [ ]:
# SHAP feature importance from step 2
result_clf.selector.importance_summary()

In [ ]:
# Build and inspect the deployment artifact
final_clf = result_clf.build_final_model()
print(final_clf)

# A DataFrame is aligned to final_clf.feature_names by column name, so the full
# feature frame can be passed as is (an ndarray must already be in that order).
X_clf = store_clf.to_frame()
print(f"Prediction sample : {final_clf.predict(X_clf)[:10]}")
print(f"P(class 1) sample : {final_clf.predict_proba(X_clf)[:5].round(3)}")

### Conformal prediction sets (classification)

`build_final_model()` automatically calibrates a conformal predictor from the out-of-fold CV predictions. `predict_set(X, alpha)` returns, for each sample, the set of classes that are plausible at the requested coverage level.

- **`[1]`** or **`[0]`** — model is confident; only one class is compatible with the coverage guarantee  
- **`[0, 1]`** — model is uncertain; both classes are plausible  

Coverage is guaranteed to be ≥ `1 - alpha` on average over new draws from the training distribution.

In [ ]:
# 90% conformal prediction sets
sets = final_clf.predict_set(X_clf, alpha=0.10)

n_confident = sum(len(s) == 1 for s in sets)
n_uncertain = sum(len(s) == 2 for s in sets)
n_empty = sum(len(s) == 0 for s in sets)

print(f"Confident   (singleton set): {n_confident} / {len(sets)}")
print(f"Uncertain   (both classes) : {n_uncertain} / {len(sets)}")
print(f"Empty set                  : {n_empty} / {len(sets)}")
print(f"\nFirst 10 prediction sets: {[s.tolist() for s in sets[:10]]}")
print(f"Calibration threshold q  : {final_clf.conformal.threshold(0.10):.4f}")

### Diagnostic plots

In [ ]:
pipeline_scores(result_clf)
cv_diagnostics(result_clf.best_cv_result)
shap_importance(result_clf.selector)
shap_summary_plot(result_clf)
shap_dependence(result_clf, n_features=6)

---
## 2. Regression — diabetes dataset with y-transform sweep

In [ ]:
data_r = load_diabetes()

store_reg = PipelineData(
    X=data_r.data.astype(np.float32),
    feature_names=list(data_r.feature_names),
    y=data_r.target.astype(np.float32),
)

print(store_reg)

In [ ]:
config_reg = PipelineConfig(
    task_type="regression",
    metric="R2",
    n_splits=5,
    n_trials=30,
    verbose=True,
)

pipeline_reg = H2MLPipeline(config=config_reg)

# Sweep log, sqrt, and identity transforms alongside raw y
result_reg = pipeline_reg.run(store_reg, transforms=["count", "log", "sqrt"])

### Results

For regression, selection in steps 1, 3 and 4 uses the **pooled** out-of-fold score (`R2_Test_Pooled`: one score over all OOF rows) by default, rather than the mean of per-fold scores (`PipelineConfig(selection_scoring="fold_mean")` switches back). Sort by the pooled column to see the ranking the pipeline actually used.

In [ ]:
result_reg.summary(metric="R2_Test_Pooled")

In [ ]:
print(f"Best model     : {result_reg.best_model_name}")
print(f"Best stage     : {result_reg.best_stage}")
print(f"Best transform : {result_reg.y_transform}")
print(f"Best params    : {result_reg.best_params}")

### Conformal prediction intervals (regression)

`predict_interval(X, alpha)` returns symmetric `(lower, upper)` bounds around the point estimate. The interval width is constant (`2q`) — the same threshold `q` applies to every prediction.

**Note:** if a y-transform was used, `predict()` and `predict_interval()` both return the original scale, so don't apply the inverse transform to their output.

In [ ]:
final_reg = result_reg.build_final_model()

X_reg = store_reg.to_frame()   # aligned to final_reg.feature_names by name

# 90% prediction intervals
lower, upper = final_reg.predict_interval(X_reg, alpha=0.10)
y_hat = final_reg.predict(X_reg)

print(f"Transform in use        : {result_reg.y_transform}")
print(f"Calibration threshold q : {final_reg.conformal.threshold(0.10):.4f}")
print(f"Interval half-width     : ±{(upper - lower).mean() / 2:.4f}")
print()

print("Sample intervals (first 5 rows):")
for i in range(5):
    print(f"  ŷ={y_hat[i]:.2f}  [{lower[i]:.2f}, {upper[i]:.2f}]")

### Diagnostic plots

In [ ]:
pipeline_scores(result_reg)
cv_diagnostics(result_reg.best_cv_result)
shap_importance(result_reg.selector)
shap_summary_plot(result_reg)
shap_dependence(result_reg, n_features=6)

---
## 3. Spatio-temporal — spatial CV & local conformal

When `store.coords` (and optionally `store.times`) are provided, the pipeline automatically switches to **spatial cross-validation** — holding out whole geographic blocks instead of random rows, so the score isn't inflated by spatial autocorrelation. With `store.times` too, conformal calibration becomes **block × season local**: the prediction interval adapts to *where* and *when* each point falls.

Below we build a synthetic dataset over the Iberian peninsula. The model's features (`temperature`, `precip`, `elevation`) carry the learnable signal, while a latitude gradient and a seasonal term are *left out of the features on purpose* — so they surface as spatially and seasonally structured residuals that local conformal calibration picks up. Still no external data required.

In [ ]:
# Synthetic spatio-temporal regression over the Iberian peninsula bounding box.
rng = np.random.default_rng(0)
n = 800

lat = rng.uniform(36.0, 44.0, n)
lon = rng.uniform(-9.0, 3.0, n)
coords = np.column_stack([lat, lon])                       # (n, 2) lat/lon

start = np.datetime64("2021-01-01")
times = (start + rng.integers(0, 730, n).astype("timedelta64[D]")).astype("datetime64[D]")

# Seasonal phase: +1 mid-summer, -1 mid-winter
doy = (times - times.astype("datetime64[Y]")).astype(int)
season = np.sin(2 * np.pi * (doy - 80) / 365.25)

# Informative covariates -> these become the model's features
elevation   = rng.uniform(0, 2000, n)
temperature = 22 - 0.6 * (lat - 36) - 0.004 * elevation + 6 * season + rng.normal(0, 1.0, n)
precip      = rng.gamma(2.0, 1.0, n) + 0.3 * (lat - 36)
noise_a     = rng.standard_normal(n)                       # uninformative -> dropped in step 2
noise_b     = rng.standard_normal(n)

X_st = np.column_stack([temperature, precip, elevation, noise_a, noise_b]).astype(np.float32)
feature_names_st = ["temperature", "precip", "elevation", "noise_a", "noise_b"]

# Target = learnable signal from features + a latitude gradient and a seasonal term that are
# deliberately NOT features, so they surface as spatially/seasonally structured residuals.
# Heteroscedastic noise (larger in the north and in summer) is what local conformal captures.
noise_scale = 1.0 + 0.5 * (lat - 36) / 8 + 1.0 * np.clip(season, 0, None)
y_st = (
    1.8 * temperature - 1.2 * precip + 0.002 * elevation
    + 1.5 * (lat - 40.0)            # spatial gradient (not a feature)
    + 2.0 * season                 # seasonal term  (not a feature)
    + rng.normal(0, noise_scale)
).astype(np.float32)

store_st = PipelineData(
    X=X_st,
    feature_names=feature_names_st,
    y=y_st,
    coords=coords,   # (n, 2) -> activates spatial cross-validation
    times=times,     # (n,)   -> activates space-time local conformal calibration
)
print(store_st)

In [ ]:
config_st = PipelineConfig(
    task_type="regression",
    metric="R2",
    n_splits=5,
    n_trials=20,
    spatial_cv_method="block",    # deterministic block grid (vs "spcv" AHC clusters)
    n_blocks_per_fold=2,          # ~16 spatial blocks, large enough for per-season cells
    time_bin_resolution="season", # compound conformal cells binned DJF/MAM/JJA/SON
    verbose=False,
)

result_st = H2MLPipeline(config=config_st).run(store_st)

print("cv_type        :", result_st.cv_type)           # -> "spatial"
print("splitter       :", type(result_st.splitter).__name__)
print("spatial blocks :", len(np.unique(result_st.splitter.block_id_)))
print("best model     :", result_st.best_model_name)
print("best R2 pooled :", round(result_st.best_model_value, 3))   # selection score of the winner

### Spatial CV blocks

`splitter.plot()` visualises the geographic blocks the model is scored on out-of-sample: the left panel colours each point by its block, the right panel by the fold that holds that block out.

In [ ]:
# Left: each point coloured by its spatial block. Right: which fold holds out that block.
result_st.splitter.plot()

### Local conformal summary

`build_final_model()` calibrates a **space-time local** conformal predictor from the out-of-fold residuals. `local_conformal.summary()` reports, per spatial block and per season, the calibration sample count `n` and the nonconformity quantile `q` (the interval half-width that block/season would apply). The `used` column shows which fallback level each cell resolves to — `compound` (block × season), `block` (season pooled), or `global`.

In [ ]:
final_st = result_st.build_final_model()

# One 'spatial' row per block + one 'compound' row per (block, season) cell.
# bin_name follows time_bin_resolution; 'used' shows the fallback level each cell resolves to.
summary_st = final_st.local_conformal.summary(alpha=0.10)
summary_st.head(12)

### Spatio-temporally adaptive intervals

Passing `coords` and `times` to `predict_interval` makes each interval's width vary by **location** and **season** — wider where and when residuals are larger. Contrast this with the constant-width intervals in the non-spatial regression section above, where a single global `q` applies to every prediction.

In [ ]:
# Passing coords + times makes the interval width adapt by location and season
X_st_df = store_st.to_frame()   # aligned to final_st.feature_names by name

lower_st, upper_st = final_st.predict_interval(
    X_st_df, alpha=0.10, coords=store_st.coords, times=store_st.times
)
half_width = (upper_st - lower_st) / 2

print(f"interval half-width  min={half_width.min():.2f}  "
      f"mean={half_width.mean():.2f}  max={half_width.max():.2f}")
print("first 8 half-widths:", half_width[:8].round(2))

### Area of applicability

Conformal intervals and CV scores only describe rows that resemble the training data. `area_of_applicability(X)` (Meyer & Pebesma 2021) returns a **dissimilarity index** (DI) per row — the distance to the nearest training row in importance-weighted, standardised feature space — and a mask that is `True` when DI is below the threshold calibrated on the CV folds. Rows outside the AOA are extrapolations: their predictions and intervals are not backed by the CV estimate.

Below, the training rows are compared with a few made-up rows far outside the training climate (very cold, very wet, high altitude).

In [ ]:
import pandas as pd

di_train, inside_train = final_st.area_of_applicability(X_st_df)
print(f"training rows inside AOA : {inside_train.mean():.1%}  (DI threshold = {final_st.applicability.threshold:.3f})")

X_far = pd.DataFrame({
    "temperature": [-15.0, -5.0, 40.0],
    "precip":      [15.0, 10.0, 0.0],
    "elevation":   [3500.0, 3000.0, 0.0],
    "noise_a":     [0.0, 0.0, 0.0],
    "noise_b":     [0.0, 0.0, 0.0],
})
di_far, inside_far = final_st.area_of_applicability(X_far)
pd.DataFrame({"DI": di_far.round(3), "inside_AOA": inside_far, "prediction": final_st.predict(X_far).round(2)})

---
## 4. Partial run — step 1 only (quick model screening)

In [ ]:
# Useful when you just want to compare models before committing to the full pipeline
result_screen = pipeline_clf.run_step1_only(store_clf)
result_screen.step1_agg_df.sort_values("AUC_Test_Mean", ascending=False)

---
## 5. Validation & robustness

Every step selects on the same CV folds, so the winner's CV score is the best of many noisy estimates and tends to be optimistic. The tools below each answer a different question. To keep the notebook fast they use few Optuna trials; in real use keep your normal settings.

### Outer holdout — one score no selection step has seen

`holdout_frac` sets rows aside before step 1 and scores the final configuration on them once. The gap to the CV score shows how much selection inflated it. Conformal calibration then uses the holdout residuals. Use `holdout_strategy="time"` to hold out the latest rows by `store.times`, and `holdout_min_minority` to drop a holdout that holds too few rare-class rows.

In [ ]:
config_ho = PipelineConfig(
    task_type="classification",
    metric="AUC",
    n_trials=10,
    holdout_frac=0.2,
)
result_ho = H2MLPipeline(config=config_ho).run(store_clf)

print(f"dev rows        : {result_ho.features.n_samples}")
print(f"holdout rows    : {len(result_ho.holdout_idx)}  ({result_ho.holdout_strategy})")
print(f"CV score        : {result_ho.best_model_value:.4f}")
print(f"holdout score   : {result_ho.holdout_df['AUC_Test_Mean'].iloc[0]:.4f}")
result_ho.summary()   # includes a "holdout" row

### Seed stability — how much does the run depend on its seed?

`random_state` decides the CV folds, SHAP sampling, the Optuna sampler and the holdout split. `seed_stability` reruns the pipeline under several seeds. If the score spread across seeds is comparable to the gaps in `ranking_robustness()`, or `Top_Selection_Fraction` is well below 1, the single-run winner is partly luck. `through_step=1` screens models only (cheapest); the default `3` also runs feature selection.

In [ ]:
from h2ml import seed_stability

stab = seed_stability(H2MLPipeline(config=config_clf), store_clf, seeds=range(3), through_step=1)

print(stab.summary().T)
stab.runs_df

### Nested CV — how well does the whole procedure generalise?

`nested_cv` runs the full pipeline inside each outer fold and scores its final configuration on the outer test fold. `Optimism_Mean` (positive = inner CV too optimistic) estimates selection bias, and `selection_counts()` / `feature_frequency()` show how stable the pipeline's choices are. It does not produce a model to deploy — it tells you what to expect from `run()` on all the data. Cost is about `outer_splits` × one `run()`.

In [ ]:
from h2ml import nested_cv

config_nested = PipelineConfig(task_type="classification", metric="AUC", n_trials=10)
nested = nested_cv(H2MLPipeline(config=config_nested), store_clf, outer_splits=3)

print(nested.summary().T)
print()
print(nested.selection_counts())
nested.folds_df

### Comparing selection metrics

The metric decides the winner, so a different metric can pick a different model. `multi_metric` runs the pipeline once per metric and shares the work that does not depend on it (step-1 CV, and steps 2–3 per step-1 winner). Choose the metric that matches how predictions will be used: Brier or LogLoss for probabilities, AUC or AUCPR for ranking, F1 for thresholded decisions.

In [ ]:
from h2ml import multi_metric

config_mm = PipelineConfig(task_type="classification", n_trials=10)
mm = multi_metric(H2MLPipeline(config=config_mm), store_clf, metrics=["Brier", "AUC"])

print(mm.summary().T)
mm.runs_df
# Each entry of mm.results is a normal PipelineResult, e.g. mm.results["Brier"].build_final_model()

Regression works the same way, with the regression metrics and an optional y-transform sweep, so each metric can also pick a different transform. `reuse_step4="model"` runs HPO once per distinct winning model instead of once per metric (`Step4_shared_with` in `runs_df` shows which runs shared it). `"PoissonDev"` can be added for non-negative targets.

In [ ]:
config_mm_reg = PipelineConfig(task_type="regression", n_trials=10)
mm_reg = multi_metric(
    H2MLPipeline(config=config_mm_reg),
    store_reg,
    metrics=["RMSE", "R2", "MAE"],
    transforms=["count", "log"],
    reuse_step4="model",
)

print(mm_reg.summary().T)
mm_reg.runs_df

---
## 6. Persistence

In [ ]:
from pathlib import Path

out = Path("outputs")
out.mkdir(exist_ok=True)

# Save the full pipeline result
result_clf.save(out / "clf_result")

# Save the deployment model
final_clf.save(out / "clf_final_model.pkl")

# Reload
from h2ml import PipelineResult  # noqa: E402
from h2ml.pipeline.final_model import FinalModel  # noqa: E402

result_reloaded = PipelineResult.load(out / "clf_result")
model_reloaded = FinalModel.load(out / "clf_final_model.pkl")

print(result_reloaded)
print(model_reloaded)